# 스포츠로만 배운 모델에 날씨를 넣으면? — 톤이 따라올까
내 모델은 **스포츠 리포트만** 보고 MBC 앵커 리드 말투를 배웠어요. 한 번도 본 적 없는 **날씨**를 넣어도 그 말투가 나올까요?

| 부분 | 넣는 것 | 정답 | 보는 점 |
|---|---|---|---|
| ① 날씨 리포트 | MBC 뉴스데스크 날씨 리포트 5건 (`test_weather.jsonl`) | 실제 앵커 리드 | 주제가 바뀌어도 리드 형식·말투가 유지되나 |
| ② 날씨 데이터 | Open-Meteo 실시간 예보 숫자 | 없음 | 숫자만 주고 기사를 쓰게 하면? 숫자를 틀리지 않나 |

준비: 4번 노트북과 같아요 (Ollama에 `qwen2.5:3b`·내 모델, `.env`의 `OPENAI_API_KEY`). `test_weather.jsonl`을 이 폴더에 넣으세요.

In [ ]:
# ▶ 1. 환경 준비 · 공통 함수 — 여기만 고치세요
import os, re, json, html, requests
from datetime import datetime
from pathlib import Path
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

MY_MODEL = 'mbcsports'      # ollama list 에 나온 내 모델 이름
RERUN    = False            # True: 저장된 답을 무시하고 다시 묻기

load_dotenv()
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
cloud = OpenAI(api_key=os.getenv('OPENAI_API_KEY')) if os.getenv('OPENAI_API_KEY') else None
MODELS = {'원본 Qwen2.5-3B': (local, 'qwen2.5:3b'),
          '내가 학습한 모델': (local, MY_MODEL),
          'gpt-4.1-mini':    (cloud, 'gpt-4.1-mini')}
COLORS = {'원본 Qwen2.5-3B': '#9aa5b1', '내가 학습한 모델': '#1f8a70', 'gpt-4.1-mini': '#2563a8'}
pd.set_option('display.max_colwidth', None)

def ask(client, model, instruction, text):
    try:
        r = client.chat.completions.create(model=model, temperature=0,
                messages=[{'role': 'user', 'content': instruction + '\n\n' + text}])
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'(실패: {type(e).__name__})'

def ask_all(df, cache):
    cache = Path(cache)
    if cache.exists() and not RERUN:
        print('저장된 답을 불러왔어요:', cache, '(다시 물으려면 RERUN = True)')
        return json.loads(cache.read_text(encoding='utf-8'))
    out = {}
    for name, (c, m) in MODELS.items():
        print('묻는 중:', name)
        out[name] = [ask(c, m, r['instruction'], r['input']) if c else '(키 없음)' for _, r in df.iterrows()]
    cache.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding='utf-8')
    print('✅ 저장:', cache)
    return out

def failed(a): return (not a) or a.startswith('(실패') or a.startswith('(키')

def checks(ans, src, gold=None):
    a = ans.replace('\\n', '\n')
    sentences = [s for s in re.split(r'(?<=[.!?다요])\s+', a.strip()) if s.strip()]
    extra = sorted(n for n in set(re.findall(r'\d[\d,.]*', a)) if n.strip('.,') not in src)
    greet = bool(re.search(r'MBC\s*뉴스\s*\S{2,4}입니다|안녕하세요', a))
    c = {'끝인사 없음': not greet, '2문장 이하': len(sentences) <= 2, '본문에 없는 숫자 없음': not extra,
         '_문장수': len(sentences), '_숫자': extra, '_인사': greet, '_글자': len(a)}
    if gold:
        c['_배율'] = len(a) / max(len(gold), 1)
        c['길이 비슷(0.5~2배)'] = 0.5 <= c['_배율'] <= 2
    return c

def summary(df, answers, gold_col='output'):
    rows = []
    for name, outs in answers.items():
        res = [checks(a, r['input'], r.get(gold_col)) for a, (_, r) in zip(outs, df.iterrows()) if not failed(a)]
        n = len(res); row = {'모델': name, '답한 문제': f'{n}/{len(outs)}'}
        for k in ['끝인사 없음', '2문장 이하', '길이 비슷(0.5~2배)', '본문에 없는 숫자 없음']:
            if res and k in res[0]: row[k] = f'{sum(x[k] for x in res)}/{n}'
        row['평균 글자 수'] = round(sum(x['_글자'] for x in res) / max(n, 1))
        rows.append(row)
    return pd.DataFrame(rows).set_index('모델')

def badges(c):
    out = [('기자 끝인사·인사말 있음' if c['_인사'] else '끝인사 없음', c['끝인사 없음']),
           (f"{c['_문장수']}문장", c['2문장 이하'])]
    if '_배율' in c:
        out.append(('길이 비슷' if c['길이 비슷(0.5~2배)'] else f"길이 {c['_배율']:.1f}배", c['길이 비슷(0.5~2배)']))
    out.append(('입력에 없는 숫자 없음' if not c['_숫자'] else '입력에 없는 숫자: ' + ', '.join(c['_숫자'][:3]), c['본문에 없는 숫자 없음']))
    return ''.join(f'<span class="b {"ok" if ok else "no"}">{html.escape(t)}</span>' for t, ok in out)

def make_report(df, answers, path, title, note, gold_col='output', src_label='리포트 본문'):
    blocks = []
    for i, (_, r) in enumerate(df.iterrows()):
        gold = r.get(gold_col) if gold_col in df.columns else None
        cards = ''
        for n in answers:
            a = answers[n][i]
            body = (f'<div class="ans empty">{html.escape(a or "답 없음")}</div>' if failed(a) else
                    f'<div class="ans">{html.escape(a.replace(chr(92)+"n", chr(10)))}</div><div class="meta">{badges(checks(a, r["input"], gold))}</div>')
            cards += (f'<div class="card"><div class="h" style="background:{COLORS.get(n, "#18232f")}">{html.escape(n)}</div>{body}'
                      '<label class="human"><input type="checkbox"> 사실 오류 있음</label><label class="human"><input type="checkbox"> MBC 리드 말투</label></div>')
        head = html.escape(r.get('title', f'문제 {i+1}'))
        gold_html = f'<div class="gold"><b>실제 앵커 리드</b> {html.escape(gold)}</div>' if gold else ''
        blocks.append(f'<h2>{i+1}. {head}</h2><details><summary>{src_label} 보기 ({len(r["input"])}자)</summary>'
                      f'<pre class="q">{html.escape(r["input"])}</pre></details>{gold_html}<div class="row">{cards}</div>')
    sum_html = summary(df, answers, gold_col).reset_index().to_html(index=False, classes='sum', border=0)
    page = f'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(title)}</title><style>
body{{font-family:"Malgun Gothic",sans-serif;margin:24px auto;max-width:1500px;background:#f7f7f5;color:#222;padding:0 16px}}
h1{{font-size:26px}} h2{{font-size:20px;margin-top:40px;border-top:2px solid #ddd;padding-top:20px}}
.sum{{border-collapse:collapse;background:#fff;margin:10px 0}} .sum th,.sum td{{border:1px solid #ddd;padding:6px 12px;text-align:center}} .sum th{{background:#18232f;color:#fff}}
.q{{background:#fff;border:1px solid #ddd;padding:12px;white-space:pre-wrap;border-radius:8px;line-height:1.6}}
.gold{{background:#fff7ed;border-left:6px solid #e8590c;padding:10px 14px;margin:10px 0;line-height:1.7}}
.row{{display:flex;gap:12px;align-items:stretch}} .card{{flex:1;background:#fff;border:1px solid #ddd;border-radius:10px;overflow:hidden;display:flex;flex-direction:column}}
.h{{color:#fff;font-weight:bold;padding:8px 12px;font-size:17px}} .ans{{white-space:pre-wrap;line-height:1.7;padding:12px;flex:1}} .empty{{color:#aaa;text-align:center}}
.meta{{padding:0 12px 8px;font-size:12px}} .b{{display:inline-block;padding:2px 8px;border-radius:10px;margin:2px 4px 2px 0}}
.ok{{background:#d8f3e8;color:#16664c}} .no{{background:#f6dede;color:#8a2a2a}}
.human{{display:block;font-size:13px;color:#555;padding:4px 12px;border-top:1px dashed #eee}} details summary{{cursor:pointer;color:#2563a8;margin:6px 0}}
</style></head><body><h1>{html.escape(title)}</h1>
<p>만든 시각: {datetime.now():%Y-%m-%d %H:%M} · 원본(회색) → 내가 학습한 모델(초록) → gpt-4.1-mini(파랑) · 초록 배지 = 리드 형식을 지킴</p>
<p>{note}</p><h2 style="border:none;margin-top:16px">자동 체크 요약</h2>{sum_html}{''.join(blocks)}</body></html>'''
    Path(path).write_text(page, encoding='utf-8')
    print('✅ 비교표 완성:', Path(path).resolve())
    if os.name == 'nt':
        os.startfile(Path(path).resolve())
print('준비 완료')

## ① MBC 날씨 리포트 5건 → 리드
학습 때와 같은 형식이에요. 지시문만 "스포츠" → "날씨"로 바꿨어요.

In [ ]:
# ▶ ①-1. 날씨 리포트 불러와서 세 모델에 묻기 (약 1분)
weather = pd.read_json('test_weather.jsonl', lines=True)
weather['instruction'] = '날씨 리포트 본문을 읽고, 앵커가 읽을 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'
ans_w = ask_all(weather, 'answers_weather.json')
pd.DataFrame({'실제 앵커 리드': weather['output'], **ans_w}, index=weather['title'])

In [ ]:
# ▶ ①-2. 자동 체크 요약 + 웹 비교표 (weather_report.html)
display(summary(weather, ans_w))
make_report(weather, ans_w, 'weather_report.html', '스포츠로 배운 모델 × 날씨 리포트',
            '내 모델은 스포츠 리포트만 학습했어요. 날씨에서도 끝인사 없이 리드 한두 문장으로 시작하는지 보세요.')

## ② Open-Meteo 날씨 데이터 → 리드
[Open-Meteo](https://open-meteo.com)는 가입 없이 쓰는 무료 날씨 API예요. 내일 예보 **숫자**만 메모로 만들어 넣어요.
정답 리드가 없으니, **메모에 없는 숫자를 지어내지 않았는지**가 가장 중요한 체크예요.

In [ ]:
# ▶ ②-1. Open-Meteo에서 내일 예보 받아 '날씨 메모' 만들기
CITIES = {'서울': (37.5665, 126.9780), '강릉': (37.7519, 128.8761), '대전': (36.3504, 127.3845),
          '부산': (35.1796, 129.0756), '제주': (33.4996, 126.5312)}
WMO = {0: '맑음', 1: '대체로 맑음', 2: '구름 조금', 3: '흐림', 45: '안개', 48: '안개', 51: '이슬비', 53: '이슬비', 55: '이슬비',
       61: '비', 63: '비', 65: '강한 비', 71: '눈', 73: '눈', 75: '강한 눈', 80: '소나기', 81: '소나기', 82: '강한 소나기', 95: '뇌우'}

def forecast(lat, lon):
    url = 'https://api.open-meteo.com/v1/forecast'
    p = {'latitude': lat, 'longitude': lon, 'timezone': 'Asia/Seoul', 'forecast_days': 2, 'wind_speed_unit': 'ms',
         'daily': 'weather_code,temperature_2m_max,temperature_2m_min,precipitation_probability_max,precipitation_sum,wind_speed_10m_max'}
    return requests.get(url, params=p, timeout=15).json()['daily']

lines, rows = [], []
for city, (lat, lon) in CITIES.items():
    d = forecast(lat, lon); i = 1                     # 0 = 오늘, 1 = 내일
    t_min, t_max = round(d['temperature_2m_min'][i]), round(d['temperature_2m_max'][i])
    t_min0 = round(d['temperature_2m_min'][0])
    rows.append({'도시': city, '날씨': WMO.get(d['weather_code'][i], '기타'), '최저': t_min, '최고': t_max, '오늘 최저': t_min0,
                 '강수확률(%)': d['precipitation_probability_max'][i], '강수량(mm)': d['precipitation_sum'][i], '최대풍속(m/s)': round(d['wind_speed_10m_max'][i])})
    lines.append(f"{city}: {WMO.get(d['weather_code'][i], '기타')}, 아침 최저 {t_min}도(오늘 {t_min0}도), 낮 최고 {t_max}도, "
                 f"강수확률 {d['precipitation_probability_max'][i]}%, 예상 강수량 {d['precipitation_sum'][i]}mm, 최대 풍속 초속 {round(d['wind_speed_10m_max'][i])}m")
date = d['time'][1]
memo = f"[{date} 내일 날씨 예보 · 출처: Open-Meteo]\n" + '\n'.join(lines)
print(memo)
pd.DataFrame(rows).set_index('도시')

In [ ]:
# ▶ ②-2. 메모로 세 모델에 리드 쓰게 하기 + 웹 비교표 (meteo_report.html)
meteo = pd.DataFrame([{'title': f'{date} 내일 날씨 (Open-Meteo)', 'input': memo,
                       'instruction': '아래 날씨 예보 메모를 읽고, 앵커가 읽을 내일 날씨 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'}])
ans_m = ask_all(meteo, f'answers_meteo_{date}.json')
for n, a in ans_m.items():
    print(f'■ {n}\n{a[0]}\n')
make_report(meteo, ans_m, 'meteo_report.html', '스포츠로 배운 모델 × 날씨 데이터',
            '정답 리드가 없어요. 메모에 없는 숫자를 지어내지 않았는지, MBC 리드 말투인지 보세요.', gold_col='output', src_label='날씨 메모')

## 무엇을 봐야 할까
| 질문 | ① 날씨 리포트 | ② 날씨 데이터 |
|---|---|---|
| 내 모델이 **끝인사 없이** 리드로 시작하나? | 스포츠에서 배운 형식이 날씨에도 나오나 | 데이터만 줘도 나오나 |
| 원본과 **말투**가 다른가? | 원본은 "MBC 뉴스 ○○○입니다"로 시작하나 | |
| **숫자**가 맞나? | 본문과 대조 | 메모의 기온·강수확률과 대조 |
| 스포츠 말투가 **엉뚱하게** 섞이나? | "승리", "연승" 같은 표현 | |

⚠ `answers_*.json`, `*_report.html`, `test_weather.jsonl`에는 MBC 기사 내용이 들어 있어요. GitHub에 올리지 않아요.